<center>


# **Setup medallion-lakehouse**

</center>

This notebook sets up the medallion-lakehouse. The only required action is to run this code in a notebook within the target Fabric Workspace.

## The easiest way to set up the metadata-driven lakehouse:

1. Create a workspace in Microsoft Fabric.
2. Create a new notebook.
3. Copy the code in the cell below into the notebook created in step 2.
4. Run the new notebook.

## Another way:

1. Create a workspace in Microsoft Fabric.
2. Download the notebook by clicking Download in GitHub.
3. Import the notebook into the target Microsoft Fabric Workspace.
4. Run the code.

![image.png](attachment:image.png)

In [ ]:
%pip install "semantic-link-labs" "PyJWT>=2.6.0" --quiet

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

LAKEHOUSE_NAMES = [
    {"Name": "bronze", "ENABLE_LAKEHOUSE_SCHEMAS": 1},
    {"Name": "silver", "ENABLE_LAKEHOUSE_SCHEMAS": 1},
    {"Name": "gold",   "ENABLE_LAKEHOUSE_SCHEMAS": 1},
]

WAREHOUSE_NAMES = []

ENABLE_COMPUTER_VISION = False

GITHUB_ACCOUNT = "https://raw.githubusercontent.com/ChristianHenrikReich/TheDataEngineersGuideToMicrosoftFabric/refs/heads/main/chapter_11/metadata_driven_lakehouse"

# ---------------------------------------------------------------------------
# Optional: Azure AI Vision integration for the count_cows_from_overview
# custom transform. When ENABLE_COMPUTER_VISION is False (default), the cow
# camera PNGs are not seeded and the count_cows_from_overview entity is
# stripped from the seeded meta_data.json. When True, fill in the service
# key and endpoint below; Setup stamps them into the parameters cell of the
# medallion-lakehouse_custom_transforms notebook at import time.
#
# Note: the stamped values live inside the imported notebook definition.
# Use a trial/workshop key, and rotate it when done.
# ---------------------------------------------------------------------------
COMPUTER_VISION_SERVICE_KEY = "<insert-from-Microsoft-ai_foundry>"
COMPUTER_VISION_ENDPOINT = "<insert-endpoint-from-Microsoft-ai_foundry>"

# Data files to seed into the bronze lakehouse. Cow camera PNGs are seeded
# into their Hive-partitioned tree year=YYYY/month=MM/day=DD so the image
# reader in Bronze to Silver can pick up the partition columns automatically.
DATA_FILES = [
    (f"{GITHUB_ACCOUNT}/data/meta-data/meta_data.json",                     "/Files/meta/meta_data.json"),
    (f"{GITHUB_ACCOUNT}/data/Sales/Customers/Customers.csv",                "/Files/data/Sales/Customers/Customers.csv"),
    (f"{GITHUB_ACCOUNT}/data/Sales/Products/Products.parquet",              "/Files/data/Sales/Products/Products.parquet"),
    (f"{GITHUB_ACCOUNT}/data/Sales/Transactions/Transactions.csv",          "/Files/data/Sales/Transactions/Transactions.csv"),
]

# Cow camera PNGs are only seeded when ENABLE_COMPUTER_VISION is True,
# since they are only useful to the count_cows_from_overview custom transform.
COW_CAMERA_DATA_FILES = [
    (f"{GITHUB_ACCOUNT}/data/Sales/Cow_Camera_Overviews/year=2025/month=01/day=03/Field_0001.png", "/Files/data/Sales/Cow_Camera_Overviews/year=2025/month=01/day=03/Field_0001.png"),
    (f"{GITHUB_ACCOUNT}/data/Sales/Cow_Camera_Overviews/year=2025/month=01/day=03/Field_0002.png", "/Files/data/Sales/Cow_Camera_Overviews/year=2025/month=01/day=03/Field_0002.png"),
    (f"{GITHUB_ACCOUNT}/data/Sales/Cow_Camera_Overviews/year=2025/month=01/day=03/Field_0003.png", "/Files/data/Sales/Cow_Camera_Overviews/year=2025/month=01/day=03/Field_0003.png"),
]

if ENABLE_COMPUTER_VISION:
    DATA_FILES = DATA_FILES + COW_CAMERA_DATA_FILES

# (github path, data source spec):
#   default_lakehouse        = lakehouse that the notebook is attached to as default
#   additional_data_sources  = extra lakehouses / warehouses the notebook can read/write
# Utility notebooks loaded via %run do not need a default lakehouse.
NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES = [
    (f"{GITHUB_ACCOUNT}/notebooks/medallion-lakehouse_functions.ipynb",                       {"default_lakehouse": None,     "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/medallion-lakehouse_custom_transforms.ipynb",               {"default_lakehouse": None,     "additional_data_sources": []}),
    (f"{GITHUB_ACCOUNT}/notebooks/1_medallion-lakehouse_Bronze_To_Silver.ipynb",              {"default_lakehouse": "bronze", "additional_data_sources": ["silver"]}),
    (f"{GITHUB_ACCOUNT}/notebooks/2_medallion-lakehouse_Silver_To_Gold.ipynb",                {"default_lakehouse": "silver",   "additional_data_sources": ["gold"]}),
    (f"{GITHUB_ACCOUNT}/notebooks/3_medallion-lakehouse_Load_Dimension_Dates.ipynb",          {"default_lakehouse": "silver",   "additional_data_sources": ["gold"]}),
    (f"{GITHUB_ACCOUNT}/notebooks/3_medallion-lakehouse_Load_Dimension_Sales_Customers.ipynb",{"default_lakehouse": "silver",   "additional_data_sources": ["gold"]}),
    (f"{GITHUB_ACCOUNT}/notebooks/3_medallion-lakehouse_Load_Dimension_Sales_Products.ipynb", {"default_lakehouse": "silver",   "additional_data_sources": ["gold"]}),
    (f"{GITHUB_ACCOUNT}/notebooks/4_medallion-lakehouse_Load_Fact_Sales_Transactions.ipynb",  {"default_lakehouse": "silver",   "additional_data_sources": ["gold"]}),
]

MAX_WORKERS = 5

# ---------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor
import json as _json
import os
import time
import requests
import notebookutils

def _item_id(item):
    """Extract the item id from a notebookutils info object or a plain dict."""
    if item is None:
        return None
    props = getattr(item, "properties", None)
    if isinstance(props, dict):
        for k in ("id", "Id", "workspaceId"):
            if k == "id" and k in props:
                return props[k]
            if k == "Id" and k in props:
                return props[k]
    if isinstance(item, dict):
        return item.get("id") or item.get("Id")
    return getattr(item, "id", None)


def _create_schema_enabled_lakehouse(name):
    """Create a schema-enabled lakehouse via the Fabric REST API."""
    import sempy.fabric as fabric
    client = fabric.FabricRestClient()
    workspace_id = fabric.get_workspace_id()
    client.post(
        f"v1/workspaces/{workspace_id}/lakehouses",
        json={
            "displayName": name,
            "description": f"Workshop lakehouse: {name}",
            "creationPayload": {"enableSchemas": True},
        },
    )
    # Long-running operation; poll notebookutils.get until the item is visible.
    for _ in range(30):
        try:
            return notebookutils.lakehouse.get(name=name)
        except Exception:
            time.sleep(2)
    return notebookutils.lakehouse.get(name=name)


def _create_or_get_lakehouses(lakehouse_specs):
    """Create or get each lakehouse. Each spec is a dict:
        {"Name": "", "ENABLE_LAKEHOUSE_SCHEMAS": 0|1}
    The schema flag is per-lakehouse. Warehouses ignore this entirely.
    """
    print("Get or create Lakehouses")
    lakehouses = {}
    for spec in lakehouse_specs:
        name = spec["Name"]
        enable_schemas = bool(spec.get("ENABLE_LAKEHOUSE_SCHEMAS", 0))
        try:
            info = notebookutils.lakehouse.get(name=name)
            print(f"  Found Lakehouse: {name}")
        except Exception:
            if enable_schemas:
                info = _create_schema_enabled_lakehouse(name)
                print(f"  Created schema-enabled Lakehouse: {name}")
            else:
                info = notebookutils.lakehouse.create(name)
                print(f"  Created Lakehouse: {name}")
        lakehouses[name] = info
    return lakehouses


def _create_or_get_warehouses(warehouse_names):
    if not warehouse_names:
        print("No Warehouses to create")
        return {}
    from sempy_labs.warehouse import create_warehouse, list_warehouses
    print("Get or create Warehouses")
    df = list_warehouses()
    existing = {row["Warehouse Name"]: row for _, row in df.iterrows()}
    for name in warehouse_names:
        if name in existing:
            print(f"  Found Warehouse: {name}")
        else:
            create_warehouse(warehouse=name, description=f"Workshop warehouse: {name}")
            print(f"  Created Warehouse: {name}")
    # Re-list so newly created warehouses appear with ids.
    df = list_warehouses()
    return {row["Warehouse Name"]: row for _, row in df.iterrows() if row["Warehouse Name"] in warehouse_names}


def _get_from_git(url):
    print(f"  Fetching {url}")
    r = requests.get(url)
    r.raise_for_status()
    return r


def _save_file_from_git(url, save_path):
    content = _get_from_git(url).content
    print(f"  Writing {save_path}")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as f:
        f.write(content)


def _inject_data_sources(content_text, spec, lakehouse_ids, warehouse_ids, workspace_id):
    """Rewrite the notebook content so its metadata points at the default lakehouse
    plus any additional lakehouses/warehouses listed in the spec."""
    nb = _json.loads(content_text)
    meta = nb.setdefault("metadata", {})
    # Start from a clean dependencies block so attendees see only what Setup configured.
    meta["dependencies"] = {}
    deps = meta["dependencies"]

    default_name = spec.get("default_lakehouse")
    additional   = spec.get("additional_data_sources") or []

    known_lh = []
    if default_name and default_name in lakehouse_ids:
        default_id = lakehouse_ids[default_name]
        known_lh.append({"id": default_id})
        deps["lakehouse"] = {
            "default_lakehouse": default_id,
            "default_lakehouse_name": default_name,
            "default_lakehouse_workspace_id": workspace_id,
            "known_lakehouses": known_lh,
        }

    known_wh = []
    for n in additional:
        if n in lakehouse_ids and not any(k["id"] == lakehouse_ids[n] for k in known_lh):
            known_lh.append({"id": lakehouse_ids[n]})
        elif n in warehouse_ids:
            known_wh.append({"id": warehouse_ids[n], "type": "Datawarehouse"})

    if known_wh:
        deps["warehouse"] = {"known_warehouses": known_wh}

    return _json.dumps(nb, indent=1)


def _stamp_custom_transform_parameters(content_text):
    """Rewrite the PARAMETERS code cell in medallion-lakehouse_custom_transforms.ipynb
    with the computer vision values configured in this Setup notebook. The
    parameters cell is identified by the '# PARAMETERS - stamped by Setup.ipynb'
    marker on its first source line."""
    nb = _json.loads(content_text)
    marker = "# PARAMETERS - stamped by Setup.ipynb"
    stamped_source = [
        marker + "\n",
        f"ENABLE_COMPUTER_VISION = {bool(ENABLE_COMPUTER_VISION)!r}\n",
        f"COMPUTER_VISION_SERVICE_KEY = {COMPUTER_VISION_SERVICE_KEY!r}\n",
        f"COMPUTER_VISION_ENDPOINT = {COMPUTER_VISION_ENDPOINT!r}\n",
    ]
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = cell.get("source") or []
        first = src[0] if src else ""
        if marker in first:
            cell["source"] = stamped_source
            break
    return _json.dumps(nb, indent=1)


def _save_notebook_from_git(url, spec, lakehouse_ids, warehouse_ids, workspace_id):
    content = _get_from_git(url).text
    content = _inject_data_sources(content, spec, lakehouse_ids, warehouse_ids, workspace_id)
    name = os.path.splitext(os.path.basename(url))[0]
    if name == "medallion-lakehouse_custom_transforms":
        content = _stamp_custom_transform_parameters(content)
    default_lh = spec.get("default_lakehouse") or ""
    try:
        notebookutils.notebook.create(name=name, content=content, defaultLakehouse=default_lh)
        print(f"  Imported notebook: {name} (default: {default_lh or 'none'})")
    except Exception:
        notebookutils.notebook.updateDefinition(name=name, content=content, defaultLakehouse=default_lh)
        print(f"  Updated notebook:  {name} (default: {default_lh or 'none'})")


# ---------------------------------------------------------------------------
# 1. Create / get the medallion lakehouses and the gold warehouse
# ---------------------------------------------------------------------------
lakehouses = _create_or_get_lakehouses(LAKEHOUSE_NAMES)
warehouses = _create_or_get_warehouses(WAREHOUSE_NAMES)

# Resolve ids for metadata injection.
import sempy.fabric as _fabric
WORKSPACE_ID = _fabric.get_workspace_id()
LAKEHOUSE_IDS = {n: _item_id(info) for n, info in lakehouses.items()}
WAREHOUSE_IDS = {}
for n, row in warehouses.items():
    # sempy_labs.list_warehouses columns include "Warehouse Name" and "Warehouse Id" (or similar)
    wid = row.get("Warehouse Id") if hasattr(row, "get") else None
    if wid is None:
        for k in ("Id", "id", "WarehouseId"):
            if k in row:
                wid = row[k]
                break
    WAREHOUSE_IDS[n] = wid

# ---------------------------------------------------------------------------
# 2. (Optional) mount a lakehouse and seed data files from GitHub
# ---------------------------------------------------------------------------
if DATA_FILES:
    primary = LAKEHOUSE_NAMES[0]["Name"]
    mount_name = primary.lower()
    print(f"Mounting {primary} to /{mount_name}")
    notebookutils.fs.mount(
        lakehouses[primary].properties["abfsPath"],
        f"/{mount_name}",
        {"fileCacheTimeout": 0, "timeout": 120},
    )
    mount_path = notebookutils.fs.getMountPath(f"/{mount_name}")
    for url, dest in DATA_FILES:
        _save_file_from_git(url, f"{mount_path}{dest}")

    # When computer vision is disabled, strip the cow camera entity out of
    # the seeded meta_data.json so the Bronze to Silver pipeline does not
    # attempt to process it.
    if not ENABLE_COMPUTER_VISION:
        meta_path = f"{mount_path}/Files/meta/meta_data.json"
        with open(meta_path) as _mf:
            _meta = _json.load(_mf)
        _filtered = [
            e for e in _meta
            if e.get("custom_transform") != "count_cows_from_overview"
        ]
        if len(_filtered) != len(_meta):
            with open(meta_path, "w") as _mf:
                _json.dump(_filtered, _mf, indent=4)
            print(f"  Stripped count_cows_from_overview entity from {meta_path}")


# ---------------------------------------------------------------------------
# 3. Import workshop notebooks into the workspace (parallel)
# ---------------------------------------------------------------------------
print("Importing workshop notebooks")
def _do_import(t):
    url, spec = t
    _save_notebook_from_git(url, spec, LAKEHOUSE_IDS, WAREHOUSE_IDS, WORKSPACE_ID)

if MAX_WORKERS == 1:
    for t in NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES:
        _do_import(t)
else:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        list(ex.map(_do_import, NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES))

print("Provisioning complete")
